# Vault Batch Processor — Colab Edition

Batch download and diarize YouTube videos using Whisper + NeMo MSDD.
Outputs one JSON file per video. Download the zip and run the import script locally to load into vault.db.

**GPU required** — Runtime > Change runtime type > T4 GPU

## 1. Install Dependencies

Run this cell once. It will **restart the runtime** at the end to load the correct numpy version. After restart, skip this cell and run from **Cell 2** onwards.

In [ ]:
!pip install -q "numpy<2.0"
!pip install -q "faster-whisper>=1.1.0" yt-dlp
!pip install -q "nemo-toolkit[asr]>=2.dev"
!pip install -q git+https://github.com/MahmoudAshraf97/demucs.git
!pip install -q git+https://github.com/oliverguhr/deepmultilingualpunctuation.git
!pip uninstall -y nvidia-cudnn-cu12 2>/dev/null

# Restart runtime to pick up the numpy downgrade — run this cell, then re-run from cell 2
import os
os.kill(os.getpid(), 9)

## 2. Configuration

Add your YouTube URLs below (one per line). Adjust whisper model and stemming as needed.

In [ ]:
import torch, os, re

# --- EDIT THESE ---
YOUTUBE_URLS = [
    # Test video (short, ~15 min — run this first to verify)
    "https://www.youtube.com/watch?v=twbDKFN1cz0",
    # Latest 10
    # "https://www.youtube.com/watch?v=e9dljIL4rBk",
    # "https://www.youtube.com/watch?v=t38LbMVoPCs",
    # "https://www.youtube.com/watch?v=Xm_PHZXGe-w",
    # "https://www.youtube.com/watch?v=s52O1JH2tnU",
    # "https://www.youtube.com/watch?v=ajgwabD4_HE",
    # "https://www.youtube.com/watch?v=EScgrk7oEwU",
    # "https://www.youtube.com/watch?v=Uvy5mcLiWW0",
    # "https://www.youtube.com/watch?v=pXlMKzcZlwM",
    # "https://www.youtube.com/watch?v=0t_DD5568RA",
    # "https://www.youtube.com/watch?v=sR7S2Q3c04g",
    # Next 10
    # "https://www.youtube.com/watch?v=99xyy1nUpug",
    # "https://www.youtube.com/watch?v=w3dTmyZq4Qk",
    # "https://www.youtube.com/watch?v=nJeU72Rgjh4",
    # "https://www.youtube.com/watch?v=yUNoJ32eLBc",
    # "https://www.youtube.com/watch?v=I_w81rptxkc",
    # "https://www.youtube.com/watch?v=FoeQUNASmTU",
    # "https://www.youtube.com/watch?v=C7LL7VwP8Nc",
    # "https://www.youtube.com/watch?v=2ZKLaUbB33o",
    # "https://www.youtube.com/watch?v=5Tr7AhkOEj4",
    # "https://www.youtube.com/watch?v=q_vZ4H3uW28",
]

WHISPER_MODEL = "large-v3"
BATCH_SIZE = 8
ENABLE_STEMMING = False  # True = better quality, uses more RAM
LANGUAGE = "en"

# --- Derived ---
OUTPUT_DIR = "/content/vault_outputs"
AUDIO_DIR = "/content/audio"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Videos to process: {len(YOUTUBE_URLS)}")
print("Uncomment the batch URLs after verifying the test video works.")

## 3. Helper Functions

Same helpers from the original whisper-diarization repo, plus download/export utilities.

In [ ]:
import json
import shutil
import subprocess
import time
import logging
import nltk
import wget
import torchaudio
import faster_whisper
from omegaconf import OmegaConf
from nemo.collections.asr.models.msdd_models import NeuralDiarizer
from deepmultilingualpunctuation import PunctuationModel

nltk.download("punkt_tab", quiet=True)

# --- YouTube helpers ---

def extract_video_id(url: str) -> str:
    """Extract video ID from a YouTube URL."""
    patterns = [
        r"(?:v=|youtu\.be/)([a-zA-Z0-9_-]{11})",
    ]
    for p in patterns:
        m = re.search(p, url)
        if m:
            return m.group(1)
    raise ValueError(f"Cannot extract video ID from: {url}")


def download_audio(video_id: str) -> tuple[str, str]:
    """Download audio as WAV via yt-dlp. Returns (wav_path, title)."""
    wav_path = os.path.join(AUDIO_DIR, f"{video_id}.wav")

    # Get title
    result = subprocess.run(
        ["yt-dlp", "--print", "%(title)s", "--no-download",
         f"https://www.youtube.com/watch?v={video_id}"],
        capture_output=True, text=True,
    )
    title = result.stdout.strip() or video_id

    if os.path.exists(wav_path):
        print(f"  Audio already exists: {video_id}.wav")
        return wav_path, title

    print(f"  Downloading audio...")
    subprocess.run(
        ["yt-dlp", "-x", "--audio-format", "wav",
         "-o", os.path.join(AUDIO_DIR, f"{video_id}.%(ext)s"),
         f"https://www.youtube.com/watch?v={video_id}"],
        check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    return wav_path, title


# --- NeMo config ---

def create_nemo_config(output_dir):
    CONFIG_URL = "https://raw.githubusercontent.com/NVIDIA/NeMo/main/examples/speaker_tasks/diarization/conf/inference/diar_infer_telephonic.yaml"
    config_path = os.path.join(output_dir, "diar_infer_telephonic.yaml")
    if not os.path.exists(config_path):
        config_path = wget.download(CONFIG_URL, output_dir)

    config = OmegaConf.load(config_path)
    data_dir = os.path.join(output_dir, "data")
    os.makedirs(data_dir, exist_ok=True)

    meta = {
        "audio_filepath": os.path.join(output_dir, "mono_file.wav"),
        "offset": 0, "duration": None, "label": "infer",
        "text": "-", "rttm_filepath": None, "uem_filepath": None,
    }
    with open(os.path.join(data_dir, "input_manifest.json"), "w") as fp:
        json.dump(meta, fp)
        fp.write("\n")

    config.num_workers = 0
    config.diarizer.manifest_filepath = os.path.join(data_dir, "input_manifest.json")
    config.diarizer.out_dir = output_dir
    config.diarizer.speaker_embeddings.model_path = "titanet_large"
    config.diarizer.oracle_vad = False
    config.diarizer.clustering.parameters.oracle_num_speakers = False
    config.diarizer.vad.model_path = "vad_multilingual_marblenet"
    config.diarizer.vad.parameters.onset = 0.8
    config.diarizer.vad.parameters.offset = 0.6
    config.diarizer.vad.parameters.pad_offset = -0.05
    config.diarizer.msdd_model.model_path = "diar_msdd_telephonic"
    return config


# --- Speaker mapping helpers (from whisper-diarization repo) ---

punct_model_langs = ["en","fr","de","es","it","nl","pt","bg","pl","cs","sk","sl"]
sentence_ending_punctuations = ".?!"


def get_word_ts_anchor(s, e, option="start"):
    if option == "end": return e
    if option == "mid": return (s + e) / 2
    return s


def get_words_speaker_mapping(wrd_ts, spk_ts, word_anchor_option="start"):
    s, e, sp = spk_ts[0]
    wrd_pos, turn_idx = 0, 0
    wrd_spk_mapping = []
    for wrd_dict in wrd_ts:
        ws = int(wrd_dict["start"] * 1000)
        we = int(wrd_dict["end"] * 1000)
        wrd = wrd_dict["text"]
        wrd_pos = get_word_ts_anchor(ws, we, word_anchor_option)
        while wrd_pos > float(e):
            turn_idx += 1
            turn_idx = min(turn_idx, len(spk_ts) - 1)
            s, e, sp = spk_ts[turn_idx]
            if turn_idx == len(spk_ts) - 1:
                e = get_word_ts_anchor(ws, we, option="end")
        wrd_spk_mapping.append({"word": wrd, "start_time": ws, "end_time": we, "speaker": sp})
    return wrd_spk_mapping


def get_first_word_idx_of_sentence(word_idx, word_list, speaker_list, max_words):
    is_end = lambda x: x >= 0 and word_list[x][-1] in sentence_ending_punctuations
    left_idx = word_idx
    while (left_idx > 0 and word_idx - left_idx < max_words
           and speaker_list[left_idx - 1] == speaker_list[left_idx]
           and not is_end(left_idx - 1)):
        left_idx -= 1
    return left_idx if left_idx == 0 or is_end(left_idx - 1) else -1


def get_last_word_idx_of_sentence(word_idx, word_list, max_words):
    is_end = lambda x: x >= 0 and word_list[x][-1] in sentence_ending_punctuations
    right_idx = word_idx
    while (right_idx < len(word_list) - 1 and right_idx - word_idx < max_words
           and not is_end(right_idx)):
        right_idx += 1
    return right_idx if right_idx == len(word_list) - 1 or is_end(right_idx) else -1


def get_realigned_ws_mapping_with_punctuation(word_speaker_mapping, max_words_in_sentence=50):
    is_end = lambda x: x >= 0 and word_speaker_mapping[x]["word"][-1] in sentence_ending_punctuations
    wsp_len = len(word_speaker_mapping)
    words_list = [d["word"] for d in word_speaker_mapping]
    speaker_list = [d["speaker"] for d in word_speaker_mapping]

    k = 0
    while k < wsp_len:
        if (k < wsp_len - 1 and speaker_list[k] != speaker_list[k + 1] and not is_end(k)):
            left_idx = get_first_word_idx_of_sentence(k, words_list, speaker_list, max_words_in_sentence)
            right_idx = get_last_word_idx_of_sentence(k, words_list, max_words_in_sentence - k + left_idx - 1) if left_idx > -1 else -1
            if min(left_idx, right_idx) == -1:
                k += 1; continue
            spk_labels = speaker_list[left_idx:right_idx + 1]
            mod_speaker = max(set(spk_labels), key=spk_labels.count)
            if spk_labels.count(mod_speaker) < len(spk_labels) // 2:
                k += 1; continue
            speaker_list[left_idx:right_idx + 1] = [mod_speaker] * (right_idx - left_idx + 1)
            k = right_idx
        k += 1

    return [dict(d, speaker=speaker_list[i]) for i, d in enumerate(word_speaker_mapping)]


def get_sentences_speaker_mapping(word_speaker_mapping, spk_ts):
    sentence_checker = nltk.tokenize.PunktSentenceTokenizer().text_contains_sentbreak
    s, e, spk = spk_ts[0]
    prev_spk = spk
    snts = []
    snt = {"speaker": f"Speaker {spk}", "start_time": s, "end_time": e, "text": "", "words": []}

    for wrd_dict in word_speaker_mapping:
        wrd, spk = wrd_dict["word"], wrd_dict["speaker"]
        s, e = wrd_dict["start_time"], wrd_dict["end_time"]
        if spk != prev_spk or sentence_checker(snt["text"] + " " + wrd):
            snts.append(snt)
            snt = {"speaker": f"Speaker {spk}", "start_time": s, "end_time": e, "text": "", "words": []}
        else:
            snt["end_time"] = e
        snt["text"] += wrd + " "
        snt["words"].append({"text": wrd, "start": s / 1000.0, "end": e / 1000.0, "score": wrd_dict.get("score")})
        prev_spk = spk
    snts.append(snt)
    return snts


def sentences_to_vault_json(sentences):
    """Convert sentence mappings to vault-compatible JSON segments."""
    segments = []
    for snt in sentences:
        if not snt["text"].strip():
            continue
        segments.append({
            "speaker": snt["speaker"],
            "start": snt["start_time"] / 1000.0,
            "end": snt["end_time"] / 1000.0,
            "text": snt["text"].strip(),
            "words": snt.get("words", []),
        })
    return segments


print("Helpers loaded.")

## 4. Process All Videos

For each video: download audio → transcribe with Whisper → diarize with NeMo MSDD → map speakers → save JSON → delete audio.

Each JSON is saved to `/content/vault_outputs/{video_id}.json` in the same format as the local pipeline.

In [ ]:
results = []  # track success/fail per video

for idx, url in enumerate(YOUTUBE_URLS, 1):
    video_id = extract_video_id(url)
    json_out = os.path.join(OUTPUT_DIR, f"{video_id}.json")

    # Skip if already processed
    if os.path.exists(json_out):
        print(f"[{idx}/{len(YOUTUBE_URLS)}] {video_id} — already done, skipping.")
        results.append((video_id, "skipped"))
        continue

    print(f"\n{'='*60}")
    print(f"[{idx}/{len(YOUTUBE_URLS)}] {video_id}")
    print(f"{'='*60}")
    t_start = time.time()

    try:
        # --- 1. Download audio ---
        wav_path, title = download_audio(video_id)
        print(f"  Title: {title}")

        # --- 2. Source separation (optional) ---
        if ENABLE_STEMMING:
            print("  Separating vocals...")
            ret = os.system(f'python -m demucs.separate -n htdemucs --two-stems=vocals "{wav_path}" -o "{AUDIO_DIR}/demucs" --device "{device}"')
            if ret == 0:
                vocal_target = os.path.join(AUDIO_DIR, "demucs", "htdemucs", video_id, "vocals.wav")
            else:
                print("  Stemming failed, using original.")
                vocal_target = wav_path
        else:
            vocal_target = wav_path

        # --- 3. Transcribe with Whisper ---
        print(f"  Loading Whisper {WHISPER_MODEL}...")
        whisper_model = faster_whisper.WhisperModel(WHISPER_MODEL, device=device, compute_type="float16")
        whisper_pipeline = faster_whisper.BatchedInferencePipeline(whisper_model)
        audio_waveform = faster_whisper.decode_audio(vocal_target)

        print(f"  Transcribing ({len(audio_waveform)/16000:.0f}s audio)...")
        transcript_segments, info = whisper_pipeline.transcribe(
            audio_waveform, LANGUAGE, batch_size=BATCH_SIZE, without_timestamps=True,
        )

        # Collect word timestamps from faster-whisper
        word_timestamps = []
        full_transcript_parts = []
        for segment in transcript_segments:
            full_transcript_parts.append(segment.text)
            if segment.words:
                for word in segment.words:
                    word_timestamps.append({
                        "text": word.word.strip(),
                        "start": word.start,
                        "end": word.end,
                        "score": word.probability,
                    })
        full_transcript = "".join(full_transcript_parts)
        print(f"  Transcribed: {len(word_timestamps)} words")

        del whisper_model, whisper_pipeline
        torch.cuda.empty_cache()

        # --- 4. NeMo MSDD Diarization ---
        print("  Running speaker diarization (NeMo MSDD)...")
        temp_path = os.path.join(AUDIO_DIR, f"temp_{video_id}")
        os.makedirs(temp_path, exist_ok=True)

        # Save mono 16kHz WAV for NeMo
        waveform_tensor = torch.from_numpy(audio_waveform).unsqueeze(0).float()
        torchaudio.save(os.path.join(temp_path, "mono_file.wav"), waveform_tensor, 16000, channels_first=True)

        msdd_model = NeuralDiarizer(cfg=create_nemo_config(temp_path)).to(device)
        msdd_model.diarize()
        del msdd_model
        torch.cuda.empty_cache()

        # Read RTTM output
        speaker_ts = []
        rttm_path = os.path.join(temp_path, "pred_rttms", "mono_file.rttm")
        with open(rttm_path, "r") as f:
            for line in f:
                parts = line.split(" ")
                s = int(float(parts[5]) * 1000)
                e = s + int(float(parts[8]) * 1000)
                speaker_ts.append([s, e, int(parts[11].split("_")[-1])])
        print(f"  Diarization: {len(speaker_ts)} speaker segments")

        # --- 5. Map speakers to words ---
        wsm = get_words_speaker_mapping(word_timestamps, speaker_ts, "start")

        # Restore punctuation
        if info.language in punct_model_langs:
            print("  Restoring punctuation...")
            punct_model = PunctuationModel(model="kredor/punctuate-all")
            words_list = [d["word"] for d in wsm]
            labeled_words = punct_model.predict(words_list, chunk_size=230)
            is_acronym = lambda x: re.fullmatch(r"\b(?:[a-zA-Z]\.){2,}", x)
            model_puncts = ".,;:!?"

            for word_dict, labeled_tuple in zip(wsm, labeled_words):
                word = word_dict["word"]
                if word and labeled_tuple[1] in sentence_ending_punctuations and (word[-1] not in model_puncts or is_acronym(word)):
                    word += labeled_tuple[1]
                    if word.endswith(".."): word = word.rstrip(".")
                    word_dict["word"] = word

        wsm = get_realigned_ws_mapping_with_punctuation(wsm)
        ssm = get_sentences_speaker_mapping(wsm, speaker_ts)

        # --- 6. Save JSON ---
        segments = sentences_to_vault_json(ssm)
        metadata = {
            "video_id": video_id,
            "title": title,
            "language": info.language,
            "duration": segments[-1]["end"] if segments else 0,
            "num_segments": len(segments),
            "num_speakers": len(set(s["speaker"] for s in segments)),
            "whisper_model": WHISPER_MODEL,
            "segments": segments,
        }
        with open(json_out, "w", encoding="utf-8") as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)
        print(f"  Saved: {json_out} ({len(segments)} segments)")

        # --- 7. Cleanup ---
        if os.path.exists(wav_path):
            os.remove(wav_path)
        if os.path.exists(temp_path):
            shutil.rmtree(temp_path)
        if ENABLE_STEMMING and os.path.exists(os.path.join(AUDIO_DIR, "demucs")):
            shutil.rmtree(os.path.join(AUDIO_DIR, "demucs"))

        elapsed = time.time() - t_start
        print(f"  Done in {elapsed:.0f}s")
        results.append((video_id, "success"))

    except Exception as e:
        print(f"  FAILED: {e}")
        results.append((video_id, f"failed: {e}"))
        # Cleanup on failure too
        wav_path = os.path.join(AUDIO_DIR, f"{video_id}.wav")
        if os.path.exists(wav_path):
            os.remove(wav_path)
        continue

# --- Summary ---
print(f"\n{'='*60}")
print("BATCH COMPLETE")
print(f"{'='*60}")
for vid, status in results:
    print(f"  {vid}: {status}")
success = sum(1 for _, s in results if s == "success")
print(f"\n{success} success, {len(results) - success} other, {len(results)} total")

## 5. Download Results

Zips all JSON files for download. On your local machine, put them in `data/audio/` and run the import script.

In [ ]:
import zipfile
from google.colab import files

zip_path = "/content/vault_outputs.zip"
json_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith(".json")])

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in json_files:
        zf.write(os.path.join(OUTPUT_DIR, f), f)

print(f"Zipped {len(json_files)} files → vault_outputs.zip")
for f in json_files:
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
    print(f"  {f} ({size:.0f} KB)")

files.download(zip_path)

## 6. Import Locally

After downloading the zip, run these on your local machine:

```bash
cd vault
# Unzip into a folder
mkdir -p data/colab_outputs
unzip vault_outputs.zip -d data/colab_outputs/

# Import all JSONs into vault.db
python pipeline/scripts/import_colab_json.py data/colab_outputs/

# Or import a single file
python pipeline/scripts/import_colab_json.py data/colab_outputs/twbDKFN1cz0.json
```

The import script:
- Creates the episode if it doesn't exist
- Skips episodes already marked `complete`
- Replaces existing segments on re-import
- Sets status to `complete` with duration